In [1]:
accepted_df = spark.read.table("bronze_accepted_loans")

rejected_df = spark.read.table("bronze_rejected_loans")

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 3, Finished, Available, Finished, False)

In [2]:
print("Accepted Loans")
print(f"Rows    : {accepted_df.count():,}")
print(f"Columns : {len(accepted_df.columns)}")

print()

print("Rejected Loans")
print(f"Rows    : {rejected_df.count():,}")
print(f"Columns : {len(rejected_df.columns)}")

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 4, Finished, Available, Finished, False)

Accepted Loans
Rows    : 2,260,701
Columns : 151

Rejected Loans
Rows    : 27,648,741
Columns : 9


# Dataset Summary

**Purpose:**  
Understand the structure of both Bronze datasets before designing the Silver layer.

**Datasets:** 
1. bronze_accepted_loans
2. bronze_rejected_loans

In [3]:
accepted_df.printSchema()

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 5, Finished, Available, Finished, False)

root
 |-- id: string (nullable = true)
 |-- member_id: string (nullable = true)
 |-- loan_amnt: double (nullable = true)
 |-- funded_amnt: double (nullable = true)
 |-- funded_amnt_inv: double (nullable = true)
 |-- term: string (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- installment: double (nullable = true)
 |-- grade: string (nullable = true)
 |-- sub_grade: string (nullable = true)
 |-- emp_title: string (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- annual_inc: string (nullable = true)
 |-- verification_status: string (nullable = true)
 |-- issue_d: string (nullable = true)
 |-- loan_status: string (nullable = true)
 |-- pymnt_plan: string (nullable = true)
 |-- url: string (nullable = true)
 |-- desc: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- title: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- dti: string 

In [4]:
rejected_df.printSchema()

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 6, Finished, Available, Finished, False)

root
 |-- amount_requested: double (nullable = true)
 |-- application_date: date (nullable = true)
 |-- loan_title: string (nullable = true)
 |-- risk_score: string (nullable = true)
 |-- debt_to_income_ratio: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- state: string (nullable = true)
 |-- employment_length: string (nullable = true)
 |-- policy_code: string (nullable = true)



In [5]:
from pyspark.sql.functions import col

schema_df = spark.createDataFrame(
    [(field.name, str(field.dataType), field.nullable)
     for field in accepted_df.schema.fields],
    ["Column", "Data Type", "Nullable"]
)

display(schema_df)

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e61f4838-4f39-446a-8f7b-1fd8c603209d)

In [6]:
from pyspark.sql.functions import col, count, when

def profile_nulls(df):

    row_count = df.count()

    return spark.createDataFrame(
        [
            (
                c,
                df.filter(
                    col(c).isNull() | (col(c) == "")
                ).count()
            )
            for c in df.columns
        ],
        ["Column","Null Count"]
    ).withColumn(
        "Null %",
        col("Null Count") * 100 / row_count
    )

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 8, Finished, Available, Finished, False)

In [7]:
accepted_nulls = profile_nulls(accepted_df)

display(
    accepted_nulls.orderBy(col("Null %").desc())
)

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7e5128d1-5583-48ef-a4cb-2212fcaa8b8c)

In [9]:
from pyspark.sql.functions import current_timestamp

log_df = spark.createDataFrame([
    (
        "pl_loan_portfolio_etl",
        "nb_data_profile",
        "SUCCESS",
    accepted_df.count()
    )
], ["pipeline_name", "notebook_name", "status", "rows_processed"])

log_df = log_df.withColumn("run_time", current_timestamp())

log_df.write.mode("append").saveAsTable("audit_log")

StatementMeta(, 99c99298-1c7a-4605-b89a-709fe9d7c25f, 11, Finished, Available, Finished, True)